# Exploration de l'API Greenhouse

Objectif : comprendre ce que renvoie l'API publique Greenhouse avant d'écrire le moindre code de production.

À la fin de ce notebook, vous aurez :
- vu la structure exacte d'une réponse et d'une offre ;
- testé le cas d'erreur (entreprise inexistante) ;
- enregistré une **fixture** de test, réutilisable sans aucun appel réseau.

Pensez à faire **Clear All Outputs** avant chaque commit.

In [233]:
import html
import json
from pathlib import Path

import httpx

## 1. Paramètres

Le *board token* est le dernier segment de l'adresse de la page carrières : `boards.greenhouse.io/stripe` → `stripe`.
Changez-le pour explorer d'autres entreprises.

In [234]:
greenhouse_tokens = [
    "artefactjobs",
    "artefact",
    "dataiku",
    "thefork",
    "algolia",
    "sonymusiccareersfrance",
    "housinganywhere",
    "doctolib",
    "datalog",
    "mirakl",
    "dashlane",
    "capfigroupe",
    "shifttechnology",
    "helsing",
    "anthropic",
    "airbnb",
    "databricks",
    "cloudfare",
    "mongodb",
    "grafanalabs",
    "pinterest",
    "celonis",
]

In [235]:
lever = ["mistral", "qonto", "blablacar", "veepee", "swile"]

In [236]:
BOARD_TOKEN = "doctolib"
BASE_URL = "https://boards-api.greenhouse.io/v1/boards"

url = f"{BASE_URL}/{BOARD_TOKEN}/jobs"
url

'https://boards-api.greenhouse.io/v1/boards/doctolib/jobs'

## 2. La requête

- `params={"content": "true"}` demande les descriptions complètes des offres (sans ce paramètre, on n'a que les métadonnées).
- `timeout=10` : on n'attend jamais indéfiniment un service externe. Réflexe à garder en production.
- `raise_for_status()` lève une exception si le code HTTP indique une erreur (4xx ou 5xx).

In [237]:
response = httpx.get(url, params={"content": "true"}, timeout=10)
print("Code de statut :", response.status_code)
print("Taille de la réponse :", round(len(response.content) / 1024), "Ko")
response.raise_for_status()

data = response.json()

Code de statut : 200
Taille de la réponse : 1981 Ko


## 3. La structure de la réponse

On commence toujours par le niveau le plus haut : quelles clés, quels types.

In [238]:
print("Clés de premier niveau :", list(data.keys()))
print("Nombre d'offres :", len(data["jobs"]))
print("Métadonnées :", data.get("meta"))

Clés de premier niveau : ['jobs', 'meta']
Nombre d'offres : 149
Métadonnées : {'total': 149}


In [239]:
data

{'jobs': [{'absolute_url': 'https://job-boards.greenhouse.io/doctolib/jobs/7583949003',
   'data_compliance': [{'type': 'gdpr',
     'requires_consent': True,
     'requires_processing_consent': False,
     'requires_retention_consent': True,
     'retention_period': 180,
     'demographic_data_consent_applies': True}],
   'education': 'education_optional',
   'internal_job_id': 5698924003,
   'location': {'name': 'Berlin, Berlin, Germany'},
   'metadata': [{'id': 4031212003,
     'name': 'Job Posting Country',
     'value': 'Germany',
     'value_type': 'single_select'},
    {'id': 4018295003,
     'name': 'Employment Type',
     'value': 'Unlimited Contract',
     'value_type': 'single_select'}],
   'id': 7583949003,
   'updated_at': '2026-09-02T05:38:47-04:00',
   'requisition_id': None,
   'title': 'Account Executive - Berlin (x/f/m)',
   'company_name': 'Doctolib',
   'first_published': '2026-01-13T08:37:47-05:00',
   'language': 'de',
   'application_deadline': None,
   'content'

## 4. Anatomie d'une offre

On affiche chaque champ de la première offre avec son type. Les descriptions longues sont tronquées pour rester lisibles.

In [240]:
premiere_offre = data["jobs"][2]

for cle, valeur in premiere_offre.items():
    apercu = repr(valeur)
    if len(apercu) > 100:
        apercu = apercu[:100] + "..."
    print(f"{cle:<20} {type(valeur).__name__:<8} {apercu}")

absolute_url         str      'https://job-boards.greenhouse.io/doctolib/jobs/7991025003'
data_compliance      list     [{'type': 'gdpr', 'requires_consent': True, 'requires_processing_consent': False, 'requires_retentio...
education            str      'education_optional'
internal_job_id      int      5884203003
location             dict     {'name': 'Pau, Pyrénées-Atlantiques, France'}
metadata             list     [{'id': 4031212003, 'name': 'Job Posting Country', 'value': 'France', 'value_type': 'single_select'}...
id                   int      7991025003
updated_at           str      '2026-09-15T05:19:29-04:00'
requisition_id       NoneType None
title                str      'Account Executive terrain - Pau (x/f/m)'
company_name         str      'Doctolib'
first_published      str      '2026-09-11T10:16:32-04:00'
language             str      'fr'
application_deadline NoneType None
content              str      '&lt;h2&gt;&lt;strong&gt;Vos missions dans l&#39;équipe commerciale&l

## 5. Le piège de la description

Le champ `content` contient du HTML **échappé** : les balises arrivent sous la forme `&lt;p&gt;` au lieu de `<p>`.
Il faut le décoder avec `html.unescape` avant de pouvoir l'exploiter. Ce sera une étape de votre adaptateur.

In [241]:
brut = premiere_offre.get("content", "")
print("Brut   :", brut[:200])
print()
print("Décodé :", html.unescape(brut)[:200])

Brut   : &lt;h2&gt;&lt;strong&gt;Vos missions dans l&#39;équipe commerciale&lt;/strong&gt;&lt;/h2&gt;
&lt;p&gt;La santé est l&#39;une des choses les plus précieuses que nous ayons. Or, les soignants manquent p

Décodé : <h2><strong>Vos missions dans l'équipe commerciale</strong></h2>
<p>La santé est l'une des choses les plus précieuses que nous ayons. Or, les soignants manquent parfois de temps et de moyens pour mene


## 6. Vue d'ensemble : titre, lieu, date

Remarquez que le lieu est un objet imbriqué (`location` → `name`), et que la date est une chaîne au format ISO 8601.

In [242]:
for offre in data["jobs"][:15]:
    lieu = offre.get("location", {}).get("name", "?")
    print(f"{offre['updated_at'][:10]}  {offre['title'][:55]:<55}  {lieu}")

2026-09-02  Account Executive - Berlin (x/f/m)                       Berlin, Berlin, Germany
2026-10-08  Account Executive terrain - Metz (x/f/m)                 Metz
2026-09-15  Account Executive terrain - Pau (x/f/m)                  Pau, Pyrénées-Atlantiques, France
2026-09-24  Account Executive (x/f/m) Außendienst                    Germany
2026-10-06  Account Executive (x/f/m) im Außendienst / Bergisches L  Wuppertal, North Rhine-Westphalia, Germany
2026-10-08  Account Executive (x/f/m) im Außendienst / Köln & Bonn   Köln
2026-10-01  Account Manager - Bestandskunden  (x/f/m)                Berlin, Berlin, Germany
2026-09-09  Account Manager  (x/f/m)                                 Münster; Osnabrück
2026-08-05  AI Talent Pool                                           Paris, Paris, France
2026-10-06  B2B Sales Manager (x/f/m) im Außendienst / Bergisches L  Wuppertal, North Rhine-Westphalia, Germany
2026-10-08  B2B Sales Manager (x/f/m) im Außendienst / Köln & Bonn   Köln
2026-08-05

## 7. Un premier filtre naïf sur les postes data

Un simple filtre par mots-clés sur le titre. Regardez attentivement les résultats : quels faux positifs ? quels postes manqués ?
C'est précisément le problème que votre futur filtre (testé en TDD) devra résoudre proprement.

In [243]:
MOTS_CLES = ["data", "machine learning", "ml ", "scientist", "analyst", "ai engineer"]

postes_data = [
    offre for offre in data["jobs"] if any(mot in offre["title"].lower() for mot in MOTS_CLES)
]

print(f"{len(postes_data)} offres retenues sur {len(data['jobs'])}\n")
for offre in postes_data:
    print("-", offre["title"])

23 offres retenues sur 149

- Data Quality Operations Intern (x/f/m)
- General Accountant Lead & Analyst (x/f/m)
- Lead Marketing Data Analyst (x/f/m)
- Lead Product Manager - Patient Data Collection (x/f/m)
- Medical Data Intern (x/f/m)
- Medical Data Intern (x/f/m)
- Performance projects analyst Intern (x/f/m)
- Principal Machine Learning Engineer - Applied AI & LLMs (x/f/m)
- Real-Time Analyst (x/f/m) - Nantes
- Research Scientist (x/f/m)
- Senior AI Engineer (x/f/m)
- Senior AI Engineer (x/f/m) - Phone Assistant
- Senior Data Ops engineer (x/f/m)
- Senior Machine Learning Engineer - Applied AI & LLMs (x/f/m)
- Senior Machine Learning Engineer - Orchestration - Applied AI & LLMs (x/f/m)
- Senior MLOps Engineer - Data Ingestion - Paris
- Senior ML Ops (x/f/m)
- Senior Site Reliability Engineer - Database (x/f/m)
- Senior Staff Machine Learning Engineer - Clinical - AI Teams (x/f/m)
- Staff DataOps Engineer (x/f/m)
- Staff Machine Learning Engineer - Retrieval (x/f/m)
-  Stage - Analy

## 8. Le cas d'erreur

Que renvoie l'API pour une entreprise qui n'existe pas ? Votre adaptateur devra gérer ce cas sans planter.

In [244]:
reponse_erreur = httpx.get(f"{BASE_URL}/entreprise-qui-nexiste-pas/jobs", timeout=10)
print("Code de statut :", reponse_erreur.status_code)
print("Corps :", reponse_erreur.text[:200])

Code de statut : 404
Corps : {"status":404,"error":"Job not found"}


## 9. Enregistrer une fixture de test

On ne garde que **3 offres** : la réponse complète pèse plusieurs mégaoctets, beaucoup trop pour un fichier de test versionné dans Git.
Une bonne fixture est petite, lisible, et contient juste assez de cas pour tester.

Le chemin est relatif au dossier du notebook (`notebooks/`), d'où le `..`.

In [245]:
dossier_fixtures = Path("..") / "tests" / "fixtures"
dossier_fixtures.mkdir(parents=True, exist_ok=True)

fixture = {"jobs": data["jobs"][:3], "meta": data.get("meta")}
chemin = dossier_fixtures / f"greenhouse_{BOARD_TOKEN}.json"

with open(chemin, "w", encoding="utf-8") as fichier:
    json.dump(fixture, fichier, ensure_ascii=False, indent=2)

print("Fixture enregistrée :", chemin.resolve())

Fixture enregistrée : C:\Users\User\Desktop\aicandidate\tests\fixtures\greenhouse_doctolib.json


## À vous : questions pour concevoir le modèle `Offre`

1. Quels champs de Greenhouse sont **toujours** présents ? Lesquels peuvent être vides ou absents ?
2. Le nombre d'années d'expérience demandé apparaît-il dans un champ dédié, ou seulement dans la description ? Qu'est-ce que ça implique pour votre projet ?
3. Y a-t-il un champ pour le type de contrat (CDI, CDD...) ou le télétravail ?
4. Refaites l'exercice avec l'API Lever (`https://api.lever.co/v0/postings/<entreprise>?mode=json`) : quels champs ont un équivalent, lesquels n'en ont pas ?

Vos réponses deviendront la liste des champs de votre modèle `Offre`, obligatoires et optionnels.